<a href="https://colab.research.google.com/github/sluong203/lane-change-automation/blob/master/LaneChangeHyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#upload and readcsv file
data = pd.read_csv('/content/autonomous_driving_expanded_dataset.csv')
data


,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,ego_acceleration_mps2,steering_angle_deg,yaw_rate_rads,throttle_position,brake_pressure,weather_condition,visibility_range_m,road_surface_condition,behavior_label
0,38.079472,-3.190934,6,0.675600,48.665354,0.741882,-0.012507,3.073696,80,26.231337,-1.898323,-6.714563,0.399968,0.051580,0.325341,rain,213.167374,wet,follow
1,95.120716,-0.796930,3,0.620099,49.519209,0.682938,-0.036788,3.449886,50,15.299050,0.930789,-24.890723,-0.150016,0.627191,0.361242,fog,227.492200,icy,follow
2,73.467400,10.636422,1,-0.216601,32.508906,0.849911,-0.026277,3.982308,30,7.015695,2.439927,14.601049,-0.436415,0.825701,0.161016,night,291.796782,icy,follow
3,60.267190,-4.799868,4,-0.613688,6.355965,0.215723,0.014275,4.179263,40,27.361192,-0.386392,23.101919,0.333522,0.556257,0.083264,rain,70.354785,icy,follow
4,16.445845,11.089491,13,1.343696,25.308504,0.753826,-0.044812,4.112925,40,23.701309,2.548616,-20.299001,0.414422,0.320478,0.578371,night,279.849029,icy,yield
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,89.842335,10.729680,1,1.675739,15.594384,0.194964,-0.007461,4.484782,30,19.056980,-2.447828,29.408186,-0.137741,0.768271,0.252596,fog,256.952057,dry,follow
4996,12.818720,11.925265,2,-1.675700,5.820384,0.934781,-0.030867,3.289929,60,17.126556,0.986600,3.661111,-0.064743,0.288502,0.184056,rain,468.134834,dry,stop
4997,33.456441,13.401237,2,1.656926,22.844954,0.581344,-0.041982,3.058843,30,8.742560,-2.061057,-13.764001,-0.005079,0.575361,0.203201,night,246.986400,wet,follow
4998,81.758793,-3.075360,13,-0.036771,49.215460,0.287325,0.017219,4.924092,30,25.171693,-0.280479,-10.947949,0.360910,0.066494,0.972135,clear,414.541673,wet,lane_change


Data Cleaning and Preprocessing

In [9]:
# make it so that we only deal with lane change
data_cleaned = data.copy()
data_cleaned = data_cleaned[data_cleaned["behavior_label"] == "lane_change"]

# find null values (there aren't any!)
data_cleaned.isnull().sum()
data_cleaned.head()

# drop irrelavent columns
data_cleaned = data_cleaned.drop(["throttle_position", "brake_pressure", "weather_condition",
                                  "visibility_range_m", "road_surface_condition", "behavior_label"], axis = 1)
data_cleaned.head()

,obstacle_distance_m,relative_speed_mps,num_obstacles,lane_offset_m,traffic_density_veh_per_km,risk_probability,road_curvature_1pm,road_width_m,speed_limit_kmh,ego_speed_mps,ego_acceleration_mps2,steering_angle_deg,yaw_rate_rads
9,71.099185,-2.089604,13,-1.622896,35.561364,0.780788,0.032582,3.869837,80,0.300548,2.265283,9.094087,0.443912
16,31.119982,-13.755080,8,0.981868,31.325165,0.674763,-0.012755,4.831833,50,15.398202,-0.182981,24.226679,-0.497822
25,78.732420,0.827639,10,-1.308310,42.645354,0.197064,-0.020890,3.306246,50,5.358837,2.853857,-2.368334,0.110767
26,20.767704,-7.232872,14,0.731028,35.127404,0.159869,-0.025490,3.496331,40,3.573405,-0.774038,-27.349624,0.419445
27,51.909209,-13.415199,10,-0.291188,38.700166,0.472984,-0.017574,4.572601,80,0.235055,-2.728228,9.081497,0.435627


EDA: Correlation Matrices